### prepair modules and bases settings

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn import datasets, linear_model
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, GridSearchCV
from sklearn.metrics import confusion_matrix,  classification_report, log_loss
from sklearn.preprocessing import StandardScaler
# from sklearn.tree import DecisionTreeClassifier
# from sklearn.preprocessing import PolynomialFeatures
# from sklearn.svm import SVC
# from sklearn.ensemble import RandomForestClassifier
from scipy.stats import norm
import scipy.io
import re
import itertools

import os
from os.path import join
import contextlib
from copy import deepcopy
import imp 
import time 
import sys

import pickle
from pdb import set_trace

from IPython.display import clear_output, display

In [2]:
# Add the directory containing your modules to the Python path
sys.path.append(os.path.abspath(os.path.join('..', 'ses2_modelstims')))

# load local functions
import stim_io
import stim_io_plotting
import vtc
import bvbabel

/home/jorvhar/miniconda3/envs/predlis/lib/python3.8/site-packages/nilearn/glm/__init__.py:55: FutureWarning: The nilearn.glm module is experimental. It may change in any future release of Nilearn.
  warn('The nilearn.glm module is experimental. '


In [3]:
import regression
import varpar

In [4]:
## LOADING GRID

# Load from MAT file
variables = scipy.io.loadmat('/media/jorvhar/Data8T/MRIData/timing data/grid_parameters_python.mat')

# Extract individual variables
tunsteps = variables['tunsteps']
freqstep = variables['freqstep']
subsample = variables['subsample']
mustep = variables['mustep']
muarray_bins = variables['muarray_bins']
muarray = variables['muarray']
fwhm = variables['fwhm']
octgrid = variables['octgrid']
sigmagrid = variables['sigmagrid']
pref_range = variables['pref_range']
sharp_range_fwhm = variables['sharp_range_fwhm']
sharp_range = variables['sharp_range']

## 1. Set up regresiion model
Options:

In [5]:
tonotopy_vmp = 'prf_permutations_for_s2_th.vmp'
# redo best analysis (between the two) with '_th' just to check if be want to be carefull with the sellection of voxels
# i.e. remove nonsensicle ones

In [6]:
### --- REGRESSION SAVING OPTIONS ---

# set modeltype
# modeltype = LinearRegression() #can be LinearRegression (ols), Ridge(alpha=..), Lasso(alpha=..)  etc.
modeltype = LinearRegression() 
key_ai = ['raw_scores', 'coefs', 'intercepts', 'correlation'] # what keys to median and mean across folds

# model return options
save_predict = False          # save y_pred-y
score_of_interest = 'score'   # what score to use  'score', 'raw_scores', 'coefs', 'intercepts', 'correlation'

# outlier options - #tobeimplemented
SD_lim = 3                    # remove y x sd higher then mean
remove_outliers = False       # if false dont remove sd outliers 

# what regressor variant to use
convolved = True   # use convolved dataset
resampled = True   # use scipy resampled data, instead of standard mean for downsampled data

zs=True   # zscore y
ts=True   # temporally smooth y - desired, if not too broad - idealy matching HRF (2.8/1.8=1.56)
hp=False  # highpass filter y - not wanted

# add drift regressors
dr=False   # drift regressor

### --- LOCATION OPTIONS ---

# file location
mridat_dir = '/media/jorvhar/Data8T/MRIData/PreProc'
logdat_dir = '/media/jorvhar/Data8T/MRIData/timing data/data'
vtc_dir = '/media/jorvhar/New Volume1/vtcs' # adviced to put vtc's on a (nvme) ssd while running analyses
pp_dir = lambda pp, ses : f'S{pp:02d}_SES{ses}'
betas_dir = 'Betas'

# tonotopy and mask filenames
tonotopy_vmp = 'prf_permutations_for_s2.vmp'
mask_fn = 'gm-subcortical.msk'

# fn lambda
fn = lambda pp, ses, run : f'S{pp:02d}_SES{ses}_run{run}_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc'


### --- PARTICIPANT OPTIONS ---

# session of interest
ses = 2

# variable that may be different per participant
ppz = [1,2,3,4,5,6,7,8,9,10]
n_splitsz = [6,6,5,5,5,5,5,5,5,5]               # 12 runs > 10:2cross, 6 fold, splits used per pp (for variable length option)
n_splitsz = [12,12,10,10,10,10,10,10,10,10]
n_runz = [12,12,10,10,10,10,10,10,10,10]        # number of runs
startpp = 1

### --- SET THEORIE REGRESSION MODELS ---

# for full 3 sets we need 7 sets
models = ['base',
          'adaptation',
          'prediction',
          'base_U_adaptation',
          'base_U_prediction',
          'adaptation_U_prediction',
          'base_U_adaptation_U_prediction']
# if we want to use sets we only need 3 models 
##models=['base_U_adaptation', 'prediction', 'base_U_adaptation_U_prediction']

## REGRESSORS IN MODELS ##
model_regressors = {'base':       ['raw_acti', 'onoff'], 
                    'adaptation': ['raw_adapt' ],      # adaptation
                    'prediction': ['pred_prob',   # voxelwise prior liklihood
                                   'error',       # voxelwise error
                                   'surprisal',   # global prior surprise
                                   'prec_w_surprisal',   # global prior surprise
                                   'precision'  # global precision
                                  ]  
                   } 
# if we want to add adapted activation
# model_regressors['adaptation'] += ['adapt_activ']

# set combination of regressors
model_regressors.update({'base_U_adaptation':             model_regressors['base']+
                                                          model_regressors['adaptation'],
                        'base_U_prediction':              model_regressors['base']+
                                                          model_regressors['prediction'], 
                        'adaptation_U_prediction':        model_regressors['adaptation']+
                                                          model_regressors['prediction'], 
                        'base_U_adaptation_U_prediction': model_regressors['base']+
                                                          model_regressors['adaptation']+
                                                          model_regressors['prediction']})
y_var = 'voxeltimecourse'

# select fn
pick_fn_prefix = 'ANTS_scores_pw_uw_er_lwo'


## NOTE FOR AFTER VACATION
## 1. ADD PRECISION TO PREDICTION MODEL X
## 2. ADD PER BLOCK DRIFT REGRESSORS, GRADIENTS (LINSPACE) FROM 0-1
## 3. ADD PRECISION WEIGHTED PRED ERROR?
## 4. ADD THE MOTION PARAMETERS

## COMPARE HRF SLOWER AND FASTER, FOR PP2

In [7]:
## REGRESSORS IN MODELS ##

## error model
model_regressors_error = {'base':       ['raw_acti', 'onoff'], 
                    'adaptation': ['raw_adapt' ],      # adaptation
                    'prediction': ['pred_prob',   # voxelwise prior liklihood
                                   'error',       # voxelwise error
                                   'surprisal',   # global prior surprise
                                   'prec_w_surprisal',   # global prior surprise
                                   'precision'   # global precision
                                  ]  
                   } 
model_regressors_error.update({'base_U_adaptation':        model_regressors_error['base']+
                                                          model_regressors_error['adaptation'],
                        'base_U_prediction':              model_regressors_error['base']+
                                                          model_regressors_error['prediction'], 
                        'adaptation_U_prediction':        model_regressors_error['adaptation']+
                                                          model_regressors_error['prediction'], 
                        'base_U_adaptation_U_prediction': model_regressors_error['base']+
                                                          model_regressors_error['adaptation']+
                                                          model_regressors_error['prediction']})

## error and global error model
model_regressors_error_ge = {'base':       ['raw_acti', 'onoff'], 
                    'adaptation': ['raw_adapt' ],      # adaptation
                    'prediction': ['pred_prob',   # voxelwise prior liklihood
                                   'error',       # voxelwise error
                                   'surprisal',   # global prior surprise
                                   'prec_w_surprisal',   # global prior surprise
                                   'precision',  # global precision
                                   'glob_err'
                                  ]  
                   } 
model_regressors_error_ge.update({'base_U_adaptation':        model_regressors_error_ge['base']+
                                                          model_regressors_error_ge['adaptation'],
                        'base_U_prediction':              model_regressors_error_ge['base']+
                                                          model_regressors_error_ge['prediction'], 
                        'adaptation_U_prediction':        model_regressors_error_ge['adaptation']+
                                                          model_regressors_error_ge['prediction'], 
                        'base_U_adaptation_U_prediction': model_regressors_error_ge['base']+
                                                          model_regressors_error_ge['adaptation']+
                                                          model_regressors_error_ge['prediction']})



## 2. Run regressions - per participant - per model - per gridpostion 
Run the full regressions, looping over participants, copy pasting files to a suitable ssd location, and doing the regression for the full grid.

In [8]:
## TEMP FOR LOOKING AT PRCISION WEIGHTING ##
modelregs = [model_regressors_error]
pick_fns = ['ANTS_scores_pw_uw_er_lwo_th']

for md_idx in range(len(modelregs)):
    model_regressors = modelregs[md_idx]
    pick_fn_prefix = pick_fns[md_idx]
## -- END TEMP -- ## -PICK_FN IS COMMENTED

    # loop over all participants
    for pp_idx in np.arange(ppz.index(startpp),len(ppz)):
        
        ### --- PREPARE PARTICIPANT DATA ---

        # fetch current pp vars
        pp = ppz[pp_idx]
        runz = np.arange(1,n_runz[pp_idx]+1)
        n_splits = n_splitsz[pp_idx]

        print(F'--RUNNING REGRESSION LOOP FOR PP: {pp} (runs={n_runz[pp_idx]},nr_splits={n_splits})--')

        # load stim df and tr df
        stim_df = stim_io.load_df(logdat_dir, pp, fn='processed_df_stim_v2')
        tr_df = stim_io.load_df(logdat_dir, pp, fn='processed_df_tr_v2')

        # create full path for vmp and mask
        mskpath = join(mridat_dir, pp_dir(pp,1), mask_fn)
        vmppath = join(mridat_dir, pp_dir(pp,1), betas_dir, tonotopy_vmp)

        # load full mask and convert to indeces
        _, msk = bvbabel.msk.read_msk(mskpath)
        msk = np.where(msk)

        # load vmp image
        vmp_head, vmp_img = bvbabel.vmp.read_vmp(vmppath)

        # load list of filenames at origin, and vtc filenames
        origin_fns = [join(mridat_dir, pp_dir(pp, ses), fn(pp,ses,run)) for run in runz]
        vtc_fns = [join(vtc_dir, fn(pp,ses,run)) for run in runz]

        # copy files to ssd for efficient and fast chuck processing
        stim_io.copy_files(origin_fns, vtc_fns)

        # load tonotopy vmp
        vmp_df = stim_io.vmp_add_realsigma(vmp_img, msk, mustep[0][0]) # 1. prfMU, 2. prfMU_hz, prfS, prfO


        ### --- RUN FULL REGRESSION ---

        # run full regression for current pp
        scores = regression.run_model_grid(tr_df,stim_df,vmp_df,vtc_fns,
                                           msk, vmp_img,
                                           pref_range,sharp_range,
                                           models, model_regressors,
                                           mustep, n_splits, modeltype, key_ai,
                                           save_predict=save_predict, 
                                           convolved=convolved, resampled=resampled,
                                           zs=zs, ts=ts, hp=hp, dr=dr)
        # clean up prints - for next pp
        clear_output(wait=True)

        # save scores
        if not os.path.exists(join(mridat_dir, pp_dir(pp, ses), 'Betas')):
            os.mkdir(join(mridat_dir, pp_dir(pp, ses), 'Betas'))

        # append the pickle result naming based on cleaning steps 
        pick_fn = pick_fn_prefix #'scores_prec'
        if ts: pick_fn = f'{pick_fn}_tempsmooth'
        if hp: pick_fn = f'{pick_fn}_highpass'
        if dr: pick_fn = f'{pick_fn}_drift'
        # pickle the results
        with open(join(mridat_dir, pp_dir(pp, ses), f'Betas/{pick_fn}.pickle'), 'wb') as handle:
            pickle.dump(scores, handle, protocol=pickle.HIGHEST_PROTOCOL)
        # loading of pickled results
        ###with open(join(mridat_dir, pp_dir(pp, ses), 'Betas/scores.pickle'), 'rb') as handle:
        ###    scores = pickle.load(handle)

        # clean up files where needed for next pp
        for fp in vtc_fns:
            os.remove(fp)



--RUNNING REGRESSION LOOP FOR PP: 10 (runs=10,nr_splits=10)--
Copied /media/jorvhar/Data8T/MRIData/PreProc/S10_SES2/S10_SES2_run1_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc to /media/jorvhar/New Volume1/vtcs/S10_SES2_run1_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc
Copied /media/jorvhar/Data8T/MRIData/PreProc/S10_SES2/S10_SES2_run2_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc to /media/jorvhar/New Volume1/vtcs/S10_SES2_run2_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc
Copied /media/jorvhar/Data8T/MRIData/PreProc/S10_SES2/S10_SES2_run3_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc to /media/jorvhar/New Volume1/vtcs/S10_SES2_run3_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc
Copied /media/jorvhar/Data8T/MRIData/PreProc/S10_SES2/S10_SES2_run4_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc to /media/jorvhar/New Volume1/vtcs/S10_SES2_run4_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc
Copied /media/jorvhar/Data8T/MRIData/PreProc/S10_SES2/S10_SES2_run5_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc to /media/jorvhar/New Vo

grid: 52/2400, 
        -current chuck took: 0.43 seconds
        -estimated time elapsed: 3.09 minutes of 142.56 minutes
grid: 53/2400, 
        -current chuck took: 0.54 seconds
        -estimated time elapsed: 3.10 minutes of 140.28 minutes
grid: 54/2400, 
        -current chuck took: 2.32 seconds
        -estimated time elapsed: 3.14 minutes of 139.40 minutes
grid: 55/2400, 
        -current chuck took: 2.12 seconds
        -estimated time elapsed: 3.17 minutes of 138.41 minutes
grid: 56/2400, 
        -current chuck took: 9.91 seconds
        -estimated time elapsed: 3.34 minutes of 143.02 minutes
grid: 57/2400, 
        -current chuck took: 1.01 seconds
        -estimated time elapsed: 3.35 minutes of 141.22 minutes
grid: 58/2400, 
        -current chuck took: 0.49 seconds
        -estimated time elapsed: 3.36 minutes of 139.12 minutes
grid: 59/2400, 
        -current chuck took: 10.27 seconds
        -estimated time elapsed: 3.53 minutes of 143.72 minutes
grid: 60/2400, 
       

grid: 119/2400, 
        -current chuck took: 6.52 seconds
        -estimated time elapsed: 5.87 minutes of 118.40 minutes
grid: 120/2400, 
        -current chuck took: 7.21 seconds
        -estimated time elapsed: 5.99 minutes of 119.81 minutes
grid: 121/2400, 
        -current chuck took: 0.62 seconds
        -estimated time elapsed: 6.00 minutes of 119.03 minutes
grid: 122/2400, 
        -current chuck took: 0.35 seconds
        -estimated time elapsed: 6.01 minutes of 118.17 minutes
grid: 123/2400, 
        -current chuck took: 0.51 seconds
        -estimated time elapsed: 6.02 minutes of 117.37 minutes
grid: 124/2400, 
        -current chuck took: 0.56 seconds
        -estimated time elapsed: 6.02 minutes of 116.61 minutes
grid: 125/2400, 
        -current chuck took: 3.23 seconds
        -estimated time elapsed: 6.08 minutes of 116.71 minutes
grid: 126/2400, 
        -current chuck took: 0.47 seconds
        -estimated time elapsed: 6.09 minutes of 115.93 minutes
grid: 127/2400, 

grid: 186/2400, 
        -current chuck took: 1.25 seconds
        -estimated time elapsed: 8.33 minutes of 107.52 minutes
grid: 187/2400, 
        -current chuck took: 0.42 seconds
        -estimated time elapsed: 8.34 minutes of 107.03 minutes
grid: 188/2400, 
        -current chuck took: 0.43 seconds
        -estimated time elapsed: 8.35 minutes of 106.56 minutes
grid: 189/2400, 
        -current chuck took: 2.51 seconds
        -estimated time elapsed: 8.39 minutes of 106.52 minutes
grid: 190/2400, 
        -current chuck took: 0.85 seconds
        -estimated time elapsed: 8.40 minutes of 106.15 minutes
grid: 191/2400, 
        -current chuck took: 0.36 seconds
        -estimated time elapsed: 8.41 minutes of 105.67 minutes
grid: 192/2400, 
        -current chuck took: 0.54 seconds
        -estimated time elapsed: 8.42 minutes of 105.23 minutes
grid: 194/2400, 
        -current chuck took: 1.23 seconds
        -estimated time elapsed: 8.44 minutes of 104.40 minutes
grid: 195/2400, 

grid: 256/2400, 
        -current chuck took: 0.56 seconds
        -estimated time elapsed: 10.48 minutes of 98.25 minutes
grid: 257/2400, 
        -current chuck took: 0.50 seconds
        -estimated time elapsed: 10.49 minutes of 97.94 minutes
grid: 258/2400, 
        -current chuck took: 0.38 seconds
        -estimated time elapsed: 10.49 minutes of 97.62 minutes
grid: 259/2400, 
        -current chuck took: 5.35 seconds
        -estimated time elapsed: 10.58 minutes of 98.07 minutes
grid: 260/2400, 
        -current chuck took: 4.29 seconds
        -estimated time elapsed: 10.66 minutes of 98.36 minutes
grid: 261/2400, 
        -current chuck took: 0.37 seconds
        -estimated time elapsed: 10.66 minutes of 98.04 minutes
grid: 262/2400, 
        -current chuck took: 0.43 seconds
        -estimated time elapsed: 10.67 minutes of 97.73 minutes
grid: 263/2400, 
        -current chuck took: 0.51 seconds
        -estimated time elapsed: 10.68 minutes of 97.43 minutes
grid: 264/2400, 

...

grid: 2093/2400, 
        -current chuck took: 0.67 seconds
        -estimated time elapsed: 85.88 minutes of 98.48 minutes
grid: 2094/2400, 
        -current chuck took: 0.65 seconds
        -estimated time elapsed: 85.89 minutes of 98.45 minutes
grid: 2095/2400, 
        -current chuck took: 5.83 seconds
        -estimated time elapsed: 85.99 minutes of 98.51 minutes
grid: 2096/2400, 
        -current chuck took: 0.61 seconds
        -estimated time elapsed: 86.00 minutes of 98.48 minutes
grid: 2097/2400, 
        -current chuck took: 0.65 seconds
        -estimated time elapsed: 86.01 minutes of 98.44 minutes
grid: 2098/2400, 
        -current chuck took: 2.81 seconds
        -estimated time elapsed: 86.06 minutes of 98.45 minutes
grid: 2099/2400, 
        -current chuck took: 0.76 seconds
        -estimated time elapsed: 86.07 minutes of 98.42 minutes
grid: 2100/2400, 
        -current chuck took: 2.82 seconds
        -estimated time elapsed: 86.12 minutes of 98.42 minutes
grid: 21

grid: 2160/2400, 
        -current chuck took: 5.16 seconds
        -estimated time elapsed: 88.16 minutes of 97.96 minutes
grid: 2161/2400, 
        -current chuck took: 2.52 seconds
        -estimated time elapsed: 88.20 minutes of 97.96 minutes
grid: 2162/2400, 
        -current chuck took: 0.63 seconds
        -estimated time elapsed: 88.21 minutes of 97.93 minutes
grid: 2163/2400, 
        -current chuck took: 0.62 seconds
        -estimated time elapsed: 88.22 minutes of 97.89 minutes
grid: 2164/2400, 
        -current chuck took: 0.79 seconds
        -estimated time elapsed: 88.24 minutes of 97.86 minutes
grid: 2165/2400, 
        -current chuck took: 0.60 seconds
        -estimated time elapsed: 88.25 minutes of 97.83 minutes
grid: 2166/2400, 
        -current chuck took: 3.49 seconds
        -estimated time elapsed: 88.31 minutes of 97.85 minutes
grid: 2167/2400, 
        -current chuck took: 4.62 seconds
        -estimated time elapsed: 88.38 minutes of 97.89 minutes
grid: 21

grid: 2228/2400, 
        -current chuck took: 1.30 seconds
        -estimated time elapsed: 90.12 minutes of 97.08 minutes
grid: 2229/2400, 
        -current chuck took: 1.33 seconds
        -estimated time elapsed: 90.14 minutes of 97.06 minutes
grid: 2230/2400, 
        -current chuck took: 4.13 seconds
        -estimated time elapsed: 90.21 minutes of 97.09 minutes
grid: 2231/2400, 
        -current chuck took: 2.42 seconds
        -estimated time elapsed: 90.25 minutes of 97.09 minutes
grid: 2232/2400, 
        -current chuck took: 0.75 seconds
        -estimated time elapsed: 90.26 minutes of 97.06 minutes
grid: 2233/2400, 
        -current chuck took: 0.65 seconds
        -estimated time elapsed: 90.28 minutes of 97.03 minutes
grid: 2234/2400, 
        -current chuck took: 0.84 seconds
        -estimated time elapsed: 90.29 minutes of 97.00 minutes
grid: 2235/2400, 
        -current chuck took: 2.22 seconds
        -estimated time elapsed: 90.33 minutes of 96.99 minutes
grid: 22

grid: 2295/2400, 
        -current chuck took: 0.58 seconds
        -estimated time elapsed: 92.38 minutes of 96.61 minutes
grid: 2296/2400, 
        -current chuck took: 3.85 seconds
        -estimated time elapsed: 92.45 minutes of 96.63 minutes
grid: 2297/2400, 
        -current chuck took: 2.95 seconds
        -estimated time elapsed: 92.49 minutes of 96.64 minutes
grid: 2298/2400, 
        -current chuck took: 0.74 seconds
        -estimated time elapsed: 92.51 minutes of 96.61 minutes
grid: 2299/2400, 
        -current chuck took: 0.82 seconds
        -estimated time elapsed: 92.52 minutes of 96.59 minutes
grid: 2300/2400, 
        -current chuck took: 4.57 seconds
        -estimated time elapsed: 92.60 minutes of 96.62 minutes
grid: 2301/2400, 
        -current chuck took: 0.56 seconds
        -estimated time elapsed: 92.61 minutes of 96.59 minutes
grid: 2302/2400, 
        -current chuck took: 0.65 seconds
        -estimated time elapsed: 92.62 minutes of 96.56 minutes
grid: 23

grid: 2362/2400, 
        -current chuck took: 1.14 seconds
        -estimated time elapsed: 94.20 minutes of 95.72 minutes
grid: 2363/2400, 
        -current chuck took: 0.73 seconds
        -estimated time elapsed: 94.22 minutes of 95.69 minutes
grid: 2364/2400, 
        -current chuck took: 0.59 seconds
        -estimated time elapsed: 94.23 minutes of 95.66 minutes
grid: 2365/2400, 
        -current chuck took: 4.21 seconds
        -estimated time elapsed: 94.30 minutes of 95.69 minutes
grid: 2366/2400, 
        -current chuck took: 0.92 seconds
        -estimated time elapsed: 94.31 minutes of 95.67 minutes
grid: 2367/2400, 
        -current chuck took: 0.66 seconds
        -estimated time elapsed: 94.32 minutes of 95.64 minutes
grid: 2368/2400, 
        -current chuck took: 2.29 seconds
        -estimated time elapsed: 94.36 minutes of 95.64 minutes
grid: 2369/2400, 
        -current chuck took: 1.48 seconds
        -estimated time elapsed: 94.39 minutes of 95.62 minutes
grid: 23